In [1]:
import Plib as pl
import numpy as np
import matplotlib.pyplot as plt
# For making attractive and informative statistical graph
plt.style.use('seaborn-darkgrid')
import pandas as pd
pd.options.mode.chained_assignment = None

ModuleNotFoundError: No module named 'Plib'

In [ ]:
import calendar
import datetime as dt
from dateutil.relativedelta import relativedelta
from dateutil import parser
def addVix2Future(f):
    VIX2=get_CBOE('VIX')
    VIX2=VIX2[['Close']]
    VIX2.columns=['VIX']
    VIX2 = VIX2.resample('1M', how='mean')
    VIX2['FDate']=getThirdWednesday(2019,1).strftime("%Y-%m-%d")
    for i in range(0,len(VIX2)-1):
        VIX2['FDate'].iloc[i]=getThirdWednesday(int(VIX2.index[i].year),int(VIX2.index[i].month)).strftime("%Y-%m-%d")
    f2=f
    f2['VIX']=0.0
    for i in range(0,len(f2)-1):
        try:
            #print(f2['Future'].iloc[i])
            ret1=VIX2[VIX2['FDate']==f2['Future'].iloc[i].strftime("%Y-%m-%d")]
            #print(float(ret1.iloc[0,0]))
            f2['VIX'].iloc[i]=float(ret1.iloc[0,0])
        except:
            print('')
    f2=f2.set_index('Future')
    return f2

def filterFutCols(futures):
    f1=futures[(futures['Close']!=0) & (futures['Settle']!=0) & (futures['Total Volume']!=0) & (futures['Open Interest']!=0)]
    f2=f1.reset_index(drop=True)
    f3=f2.groupby([f2['Group']],as_index=False).mean()
    f3=f3.sort_values(by='Group', ascending=True)
    f3.columns=['Future', 'Open', 'High', 'Low', 'Close', 'Settle', 'Change']
    return f3

def getThirdWednesday(year,month):
    c = calendar.Calendar(firstweekday=calendar.SUNDAY)
    monthcal = c.monthdatescalendar(year,month)
    third_friday = [day for week in monthcal for day in week if day.weekday() == calendar.WEDNESDAY and day.month == month][2]
    return third_friday

def getDatesFutures():
    today=dt.date.today()
    index=0
    df = pd.DataFrame([],columns=['Date'])
    for i in range(-12, 8):
        mydate=today + relativedelta(months=i)
        year=mydate.year
        month=mydate.month
        #print(month,year)
        df.loc[index] = getThirdWednesday(year,month).strftime("%Y-%m-%d")
        index=index+1
    df.sort_values(by='Date', ascending=True)
    return df

def get_CBOE(series):
    url_VVIX='http://www.cboe.com/publish/scheduledtask/mktdata/datahouse/vvixtimeseries.csv'
    url_VIX='http://www.cboe.com/publish/scheduledtask/mktdata/datahouse/vixcurrent.csv'
    url_VIX3='http://www.cboe.com/publish/scheduledtask/mktdata/datahouse/vix3mdailyprices.csv'
    url_VXN='http://www.cboe.com/publish/ScheduledTask/MktData/datahouse/vxncurrent.csv'
    url_SKEW='http://www.cboe.com/publish/scheduledtask/mktdata/datahouse/skewdailyprices.csv'
    url_IC='http://www.cboe.com/publish/scheduledtask/mktdata/datahouse/implied_correlation_hist.csv'
    url_FUTURES='https://markets.cboe.com/us/futures/market_statistics/historical_data/products/csv/VX/'
    if series=='FUTURES':
        data = pd.DataFrame([],columns=['Trade Date', 'Futures', 'Open', 'High', 'Low', 'Close', 'Settle',
        'Change', 'Total Volume', 'EFP', 'Open Interest','Group'])
        md=pd.DataFrame(getDatesFutures())
        md['Date']=pd.to_datetime(md['Date'])
        index=0
        for i in range(0, len(md)):
            try:
                url=url_FUTURES + str(md.iloc[i].Date.date())
                data2 = pd.read_csv(url)
                data2['Group']=md.iloc[i].Date.date()
                data=data.append(data2)
                index=index+1
            except:
                print('Downloaded all data...')
    elif series=='VVIX':
        url=url_VVIX
        data = pd.read_csv(url)
        data2=data
        data2['Date']=pd.to_datetime(data2.iloc[1:,0])
        data2['VVIX']=data2.iloc[1:,1]
        data=data2[['Date','VVIX']]
        data['VVIX']=data2['VVIX'].astype(float)
        data=data.set_index('Date')
        data=data.dropna()
    elif series=='IC':
        #DATE,SPX,VIX,(JCJ) January 2008,(KCJ) January 2009,(ICJ) January 2010,(JCJ)
        url=url_IC
        data = pd.read_csv(url)
        data2=data
        data2['Date']=data.iloc[2:,0]
        data2['Date']=pd.to_datetime(data2['Date'])
        data2['SPX']=data.iloc[2:,1]
        data2['VIX']=data.iloc[2:,2]
        data2['JCJ1']=data.iloc[2:,3]
        data2['KCJ1']=data.iloc[2:,4]
        data2['ICJ1']=data.iloc[2:,5]
        data2['JCJ2']=data.iloc[2:,6]
        data2['KCJ2']=data.iloc[2:,7]
        data2['ICJ2']=data.iloc[2:,8]
        data2['JCJ3']=data.iloc[2:,9]
        data2['KCJ3']=data.iloc[2:,10]
        data2['ICJ3']=data.iloc[2:,11]
        data2['JCJ4']=data.iloc[2:,12]
        data2['KCJ4']=data.iloc[2:,13]
        data2['ICJ4']=data.iloc[2:,14]
        data2['JCJ5']=data.iloc[2:,15]
        data2['KCJ5']=data.iloc[2:,16]
        data2=data2.replace(r'^\s*$', np.nan, regex=True)
        data3=data2[['Date','JCJ1','KCJ1','ICJ1','JCJ2','KCJ2','ICJ2','JCJ3','KCJ3','ICJ3','JCJ4','KCJ4','ICJ4','JCJ
        5','KCJ5']]
        data3=data3.set_index('Date')
        data3['JCJ1']=data3.JCJ1.astype(float)
        data3['KCJ1']=data3.KCJ1.astype(float)
        data3['ICJ1']=data3.ICJ1.astype(float)
        data3['JCJ2']=data3.JCJ2.astype(float)
        data3['KCJ2']=data3.KCJ2.astype(float)
        data3['ICJ2']=data3.ICJ2.astype(float)
        data3['JCJ3']=data3.JCJ3.astype(float)
        data3['KCJ3']=data3.KCJ3.astype(float)
        data3['ICJ3']=data3.ICJ3.astype(float)
        data3['JCJ4']=data3.JCJ4.astype(float)
        data3['KCJ4']=data3.KCJ4.astype(float)
        data3['ICJ4']=data3.ICJ4.astype(float)
        data3['JCJ5']=data3.JCJ5.astype(float)
        data3['KCJ5']=data3.KCJ5.astype(float)
        data3=pd.DataFrame(data3.mean(axis=1))
        data3['Corr']=data3.iloc[:,0]
        data3=data3[['Corr']]
        data=data2[['Date','SPX','VIX']]
        data=data.merge(data3,left_on='Date',right_on='Date',how='inner')
        data['SPX']=data.SPX.astype(float)
        data['VIX']=data.VIX.astype(float)
        data=data.set_index('Date')
        data=data.dropna()
    elif series=='SKEW':
        url=url_SKEW
        data = pd.read_csv(url)
        data2=data
        data2['Date']=pd.to_datetime(data2.iloc[1:,0])
        data2['SKEW']=data2.iloc[1:,1]
        data=data2[['Date','SKEW']]
        data['SKEW']=data2['SKEW'].astype(float)
        data=data.set_index('Date')
        data=data.dropna()
    elif series=='VIX':
        url=url_VIX
        data = pd.read_csv(url)
        data2=data
        #print(data2)
        data2['Date']=data.iloc[2:,0]
        data2['Date']=pd.to_datetime(data2['Date'])
        data2['Open']=data.iloc[2:,1]
        data2['High']=data.iloc[2:,2]
        data2['Low']=data.iloc[2:,3]
        data2['Close']=data.iloc[2:,4]
        data=data2[['Date','Open','High','Low','Close']]
        data['Open']=data.Open.astype(float)
        data['High']=data.High.astype(float)
        data['Low']=data.Low.astype(float)
        data['Close']=data.Close.astype(float)
        data=data.set_index('Date')
        data=data.dropna()
    elif series=='VIX3':
        url=url_VIX3
        data = pd.read_csv(url)
        data2=data
        data2['Date']=data.iloc[2:,0]
        data2['Date']=pd.to_datetime(data2['Date'])
        data2['Open']=data.iloc[2:,1]
        data2['High']=data.iloc[2:,2]
        data2['Low']=data.iloc[2:,3]
        data2['Close']=data.iloc[2:,4]
        data=data2[['Date','Open','High','Low','Close']]
        data['Open']=data.Open.astype(float)
        data['High']=data.High.astype(float)
        data['Low']=data.Low.astype(float)
        data['Close']=data.Close.astype(float)
        data=data.set_index('Date')
        data=data.dropna()
    elif series=='VXN':
        url=url_VXN
        data = pd.read_csv(url)
        data2=data
        data2['Date']=data.iloc[2:,0]
        data2['Date']=pd.to_datetime(data2['Date'])
        data2['Open']=data.iloc[2:,1]
        data2['High']=data.iloc[2:,2]
        data2['Low']=data.iloc[2:,3]
        data2['Close']=data.iloc[2:,4]
        data=data2[['Date','Open','High','Low','Close']]
        data['Open']=data.Open.astype(float)
        data['High']=data.High.astype(float)
        data['Low']=data.Low.astype(float)
        data['Close']=data.Close.astype(float)
        data=data.set_index('Date')
        data=data.dropna()
    return data